# Audiorium — Music Recommendation with Latent Representations of Audio

**Authors:** Anish Ghosh & Yestin Arvin Gochuico  
**Course:** Design with AI — Fall 2025, University of Miami

Implements the full Audiorium flow:

| Path | Input | Model | Output |
|---|---|---|---|
| **Search (blue)** | Text prompt | CLAP text-to-audio | Song recommendations |
| **Search (blue)** | Image upload | OpenCLIP → CLAP | Top matching song |
| **Queue (red)** | Selected song | CLAP audio-to-audio | Auto-queue (loops) |

All results pass through **Rule-Based Ordering** before being returned.

## 1. Setup & Imports

In [ ]:
# Uncomment to install
# %pip install torch torchaudio transformers faiss-cpu spotipy librosa soundfile requests open_clip_torch Pillow --quiet

In [ ]:
import os
import glob
import pickle
import numpy as np
import torch
import librosa
import faiss
from PIL import Image
from transformers import AutoProcessor, ClapModel
import open_clip
from spotipy import Spotify
from spotipy.oauth2 import SpotifyClientCredentials

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 2. Config

In [ ]:
# Works both locally (__file__ defined) and in Colab/Jupyter (getcwd fallback)
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

AUDIO_FOLDER = os.path.join(BASE_DIR, "mp3dataset")
INDEX_PATH   = os.path.join(BASE_DIR, "clap_music_index.faiss")
IDS_PATH     = os.path.join(BASE_DIR, "ids.pkl")

SPOTIFY_ID     = "################################"
SPOTIFY_SECRET = "################################"

# Music mood/genre labels for OpenCLIP zero-shot image classification
MUSIC_PROMPTS = [
    "upbeat energetic music",
    "melancholic sad music",
    "aggressive heavy rock music",
    "calm relaxing ambient music",
    "romantic love music",
    "dance electronic music",
    "acoustic folk music",
    "smooth jazz music",
    "classical orchestral music",
    "hip hop urban music",
    "dramatic cinematic music",
    "dark moody atmospheric music",
    "happy cheerful pop music",
    "intense workout music",
    "dreamy ethereal music",
]

print(f"Audio folder: {AUDIO_FOLDER}")

## 3. Load Models

- **CLAP** `laion/larger_clap_music_and_speech` — shared latent space for audio and text (text-to-audio + audio-to-audio)
- **OpenCLIP** `ViT-B-32` — vision encoder; zero-shot classifies an image against music mood labels, then hands off to CLAP (image-to-audio)

In [ ]:
clap_processor = AutoProcessor.from_pretrained("laion/larger_clap_music_and_speech")
clap_model     = ClapModel.from_pretrained("laion/larger_clap_music_and_speech").to(device)
clap_model.eval()
print("CLAP loaded")

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model     = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
print("OpenCLIP loaded")

## 4. Embedding Helpers

In [ ]:
def embed_audio(path):
    """CLAP audio encoder — maps an audio file into the CLAP latent space."""
    wav, _ = librosa.load(path, sr=48000, mono=True)
    inputs = clap_processor(audios=wav, return_tensors="pt", sampling_rate=48000).to(device)
    with torch.no_grad():
        feats = clap_model.get_audio_features(**inputs)
    return feats.cpu().numpy().squeeze()


def embed_text(texts: list):
    """CLAP text encoder — maps a list of strings into the CLAP latent space."""
    inputs = clap_processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        feats = clap_model.get_text_features(**inputs)
    return feats.cpu().numpy()


def embed_image(image_path):
    """
    OpenCLIP image-to-audio:
      1. Encode image with OpenCLIP's vision encoder.
      2. Zero-shot classify against MUSIC_PROMPTS to find the best mood label.
      3. Encode that label with CLAP to enter the audio latent space.
    Returns (clap_embedding, matched_label)
    """
    image       = clip_preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer(MUSIC_PROMPTS).to(device)

    with torch.no_grad():
        img_feats  = clip_model.encode_image(image)
        txt_feats  = clip_model.encode_text(text_tokens)
        img_feats /= img_feats.norm(dim=-1, keepdim=True)
        txt_feats /= txt_feats.norm(dim=-1, keepdim=True)
        scores     = (img_feats @ txt_feats.T).softmax(dim=-1).squeeze()

    best_label = MUSIC_PROMPTS[scores.argmax().item()]
    confidence = scores.max().item()
    print(f"  Image → '{best_label}' ({confidence:.1%} confidence)")

    return embed_text([best_label])[0], best_label

## 5. Rule-Based Ordering

Instead of returning the raw top-k FAISS hits, we:
1. Retrieve a wider candidate pool (`k × pool_factor` results)
2. Draw `k` tracks via **score-weighted random sampling**

This adds slight randomization so the same top tracks don't always appear, promoting equitable visibility of diverse and independent content.

In [ ]:
def rule_based_ordering(distances, indices, k=5, pool_factor=3):
    pool      = min(len(ids), k * pool_factor)
    pool_dist = distances[:pool]
    pool_idx  = indices[:pool]

    weights   = np.exp(pool_dist - pool_dist.max())
    weights  /= weights.sum()
    chosen    = np.random.choice(pool, size=min(k, pool), replace=False, p=weights)
    chosen    = chosen[np.argsort(-pool_dist[chosen])]  # re-sort by score

    return [(ids[pool_idx[i]], float(pool_dist[i])) for i in chosen]

## 6. Build / Load FAISS Index

Embed every track in `mp3dataset/` with CLAP and store in a `IndexFlatIP` (inner product = cosine similarity after L2 normalisation). On subsequent runs, load the saved index instead of re-embedding.

In [ ]:
def build_index(audio_folder=AUDIO_FOLDER):
    paths = (
        glob.glob(os.path.join(audio_folder, "**", "*.mp3"), recursive=True) +
        glob.glob(os.path.join(audio_folder, "**", "*.wav"), recursive=True)
    )
    print(f"Found {len(paths)} audio files — embedding now...")

    embeddings, ids_list = [], []
    for i, path in enumerate(paths):
        try:
            vec = embed_audio(path)
            embeddings.append(vec)
            ids_list.append(os.path.basename(path))
            print(f"  [{i+1}/{len(paths)}] {os.path.basename(path)}")
        except Exception as e:
            print(f"  Skipped {os.path.basename(path)}: {e}")

    emb = np.vstack(embeddings).astype("float32")
    faiss.normalize_L2(emb)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)

    faiss.write_index(idx, INDEX_PATH)
    with open(IDS_PATH, "wb") as f:
        pickle.dump(ids_list, f)
    print(f"Index saved: {idx.ntotal} tracks → {INDEX_PATH}")
    return idx, ids_list


def load_index():
    idx = faiss.read_index(INDEX_PATH)
    with open(IDS_PATH, "rb") as f:
        ids_list = pickle.load(f)
    print(f"Index loaded: {idx.ntotal} tracks")
    return idx, ids_list

In [ ]:
if os.path.exists(INDEX_PATH) and os.path.exists(IDS_PATH):
    index, ids = load_index()
else:
    index, ids = build_index()

## 7. Search Path — Text & Image

```
Text Prompt  →  CLAP text-to-audio  →  Top N  →  Rule-Based Ordering  →  Recommendations
Image Upload →  OpenCLIP → CLAP     →  Top N  →  Rule-Based Ordering  →  Top Song
```

In [ ]:
def search_by_text(prompt, k=5):
    """User Search Prompt → CLAP text-to-audio → Top N → Rule-Based Ordering → Recommendations"""
    q = embed_text([prompt])[0].astype("float32")
    faiss.normalize_L2(q.reshape(1, -1))
    distances, indices = index.search(q.reshape(1, -1), min(len(ids), k * 3))
    return rule_based_ordering(distances[0], indices[0], k=k)


def search_by_image(image_path, k=5):
    """User Uploaded Image → OpenCLIP image-to-audio → Top N → Rule-Based Ordering → Top Song"""
    q, label = embed_image(image_path)
    q = q.astype("float32")
    faiss.normalize_L2(q.reshape(1, -1))
    distances, indices = index.search(q.reshape(1, -1), min(len(ids), k * 3))
    return rule_based_ordering(distances[0], indices[0], k=k), label

In [ ]:
# Text search demo
results = search_by_text("chill hip hop song with fat 808s")
print("Query: 'chill hip hop song with fat 808s'")
for track, score in results:
    print(f"  {score:.4f}  {track}")

In [ ]:
# Image search demo — swap in any image path
# image_path = "assets/glowing_whale.HEIC"   # .jpg / .png work too
# results, label = search_by_image(image_path)
# print(f"Mood detected: '{label}'")
# for track, score in results:
#     print(f"  {score:.4f}  {track}")

## 8. Queue Path — Audio-to-Audio Auto Queue

```
User Selects a Song
  → CLAP audio-to-audio
  → Top N Songs (Track ID + Accuracy Score)
  → Rule-Based Ordering
  → Automatic Queue of Songs
       ↑________________________| (each queued song becomes the next seed)
```

In [ ]:
def build_queue(seed, queue_length=10):
    """
    Builds an automatic queue starting from a seed track.
    Each added track becomes the next seed (audio-to-audio loop).
    seed: filename (e.g. 'Plain.mp3') or a full path.
    """
    if not os.path.isfile(seed):
        matches = glob.glob(os.path.join(AUDIO_FOLDER, "**", seed), recursive=True)
        if not matches:
            raise FileNotFoundError(f"Could not find: {seed}")
        seed = matches[0]

    print(f"Queue seed: {os.path.basename(seed)}")
    queue   = []
    visited = {os.path.basename(seed)}
    current = seed

    while len(queue) < queue_length:
        q = embed_audio(current).astype("float32")
        faiss.normalize_L2(q.reshape(1, -1))
        distances, indices = index.search(q.reshape(1, -1), min(len(ids), 18))
        candidates = rule_based_ordering(distances[0], indices[0], k=6)

        next_track = None
        for track, score in candidates:
            if track not in visited:
                next_track = (track, score)
                visited.add(track)
                break

        if next_track is None:
            break

        queue.append(next_track)
        matches = glob.glob(os.path.join(AUDIO_FOLDER, "**", next_track[0]), recursive=True)
        if matches:
            current = matches[0]

    return queue

In [ ]:
# Queue demo — use any track name from mp3dataset as the seed
seed = "Plain.mp3"   # swap for any track in your mp3dataset
print(f"Seed: {seed}\n")
for track, score in build_queue(seed, queue_length=8):
    print(f"  {score:.4f}  {track}")

## 9. Full Flow — Image → Top Song → Auto Queue

In [ ]:
# Upload an image → get a top song → build a queue from it
# image_path = "assets/glowing_whale.HEIC"   # swap in your image
#
# results, label = search_by_image(image_path, k=1)
# top_song = results[0][0]
# print(f"Image mood: '{label}' → seed: {top_song}\n")
#
# for track, score in build_queue(top_song, queue_length=8):
#     print(f"  {score:.4f}  {track}")

## 10. Save Index

In [ ]:
# Index is auto-saved during build_index().
# Run this cell to manually re-save if you modified the index in-session.
faiss.write_index(index, INDEX_PATH)
with open(IDS_PATH, "wb") as f:
    pickle.dump(ids, f)
print(f"Saved: {INDEX_PATH}")